# Data Generation Walkthrough

This notebook explains how the synthetic campaign dataset is created and why the project uses this structure.

The main learning goal is to separate three layers that are easy to conflate in measurement work:

- observed campaign metrics such as impressions, clicks, conversions, and cost
- latent causal quantities such as baseline conversions and true incremental conversions
- data availability choices that determine which estimator tools the agent is allowed to use

For this project, the agent never sees the hidden truth directly. The hidden truth exists only so we can evaluate the agent later.

## Step 1: Load Or Create A Reproducible Dataset

This cell points to the default materialized dataset on disk.

Why this matters:

- using a saved dataset makes notebook results reproducible across sessions
- the same dataset can be reused by the CLI runner, the notebooks, and LangSmith traces
- if the CSV does not exist yet, we generate it once and save it so the rest of the notebook can work from a stable artifact


In [ ]:
from pathlib import Path

import pandas as pd

from iroas_agent.data import DEFAULT_DATASET_PATH, generate_campaigns, write_campaign_dataset

In [ ]:
dataset_path = Path(DEFAULT_DATASET_PATH)
if not dataset_path.exists():
    campaigns = generate_campaigns(200, seed=11)
    write_campaign_dataset(campaigns, dataset_path)

df = pd.read_csv(dataset_path)
df.head(3)

## Step 2: Inspect The Columns

This next cell helps us read the dataset schema as a modeling story.

What the columns mean:

- the plain campaign metrics are the agent-visible operational inputs
- the flattened lift-test and geo columns make the estimator-specific evidence inspectable
- the hidden truth columns are included only for education and evaluation, not for agent decision-making

Why this matters:

- we want to see clearly what information the agent can act on
- we also want to preserve the ground truth needed to study estimator bias and agent mistakes


In [ ]:
list(df.columns)

## Step 3: Compare Observed Outcomes To Hidden Truth

The histogram cell below visualizes the difference between what a marketer might observe and what the evaluator knows.

What the code does:

- plots distributions for observed conversions, true incremental conversions, and true iROAS

Why this matters:

- observed conversions combine baseline demand, ad lift, and noise
- true incremental conversions are usually much smaller than total conversions
- true iROAS is the causal target the tools are trying to approximate under imperfect information


In [ ]:
import matplotlib.pyplot as plt

df[['conversions', 'true_incremental_conversions', 'true_iROAS']].hist(figsize=(10, 4), bins=20)
plt.tight_layout()

## Step 4: Inspect Data Availability Patterns

The agent's behavior depends heavily on whether lift-test data or geo-experiment data is available.

What this cell does:

- counts how many campaigns fall into each availability bucket

Why this matters:

- it tells us how often the agent will have access to stronger causal evidence
- it also tells us how often the agent must fall back to weaker estimators


In [ ]:
df['availability_type'].value_counts()

## What To Take Away

After working through this notebook, the important mental model is:

- the project simulates a causal world and then only partially reveals it
- the dataset contains enough structure to make estimator selection meaningful
- the saved CSV is the shared starting point for the rest of the notebooks and the CLI experiments
